# Track 09 — Framework Bridges (프레임워크 브리지)

### Framework Bridge란?

이미 LangChain·LlamaIndex·LangGraph·Gradio 를 쓰는 조직에서 EXAONE 을 연동하려면 **최소 어댑터(bridge) 한 겹**이 필요합니다. EXAONE 은 OpenAI 호환 엔드포인트를 제공하므로 대부분의 프레임워크에서는 `base_url` 만 바꾸면 연결할 수 있습니다. 다만 **비표준 model id 와 EXAONE 고유 파라미터(`enable_thinking`·`top_p`)** 는 표준 OpenAI 클라이언트가 그대로 전달하지 못하므로 별도로 처리해야 합니다. 이 노트북은 그 연결 계층을 프레임워크별로 보여 줍니다.

> 이 트랙은 **선택(optional)** 입니다 — 이미 LangChain·LlamaIndex·LangGraph·Gradio 를 쓰는 환경에 EXAONE 을 연결해야 할 때의 패턴을 다룹니다. 새 에이전트를 처음 만든다면 EXAONE 라이브러리만으로 구현하는 캡스톤(Track 10)을 권장합니다.

### 이 노트북에서 보여줄 것

| Session | 프레임워크 | 무엇을 보여 주는가 | 핵심 브리지 포인트 |
|---|---|---|---|
| 1 | **LangChain** | LCEL 파이프·RAG 인용·도구 호출 1턴·thinking 누출 **대조** | `ChatOpenAI(base_url=…)` + `extra_body` 로 `enable_thinking=False` 전달 |
| 2 | **LlamaIndex** | QueryEngine 검색(점수)·ReAct 1턴 | `OpenAICompatLLM` 이 비표준 model id 수용 + `additional_kwargs`/`extra_body` 로 `top_p`·thinking 전달 |
| 3 | **LangGraph** | planner→executor→critic 흐름을 명시적 그래프로 표현(키 불필요) | 조건부 엣지로 **루프를 그래프에** 드러내고 EXAONE-native 단계와 비교 |
| 4 | **Gradio** | 노트북에서 채팅 UI 실행(URL 클릭) + 스크립트 export | `chat_stream` 을 `gr.ChatInterface` 에 연결하고 실행 파일로 내보내기 |

### 이 노트북을 마치면

- EXAONE 을 LangChain LCEL·tool calling, LlamaIndex QueryEngine·ReAct 에 **최소 어댑터로 연결**할 수 있습니다.
- 표준 OpenAI 클라이언트에서 **EXAONE 비표준 파라미터(`enable_thinking`·`top_p`)가 어떻게 누락되거나 전달되는지** 직접 확인하고(Session 1-3 대조), `extra_body`/`additional_kwargs` 로 어떻게 바로잡는지 설명할 수 있습니다.
- LangGraph 의 **명시적 그래프 루프**와 EXAONE-native ToolAgent 의 **내부 루프**의 차이를 비교할 수 있습니다.

**산출물:** `_out/langchain_vs_exaone.json`, `_out/react_trace.json`, `_out/langgraph_vs_exaone.json`, `_out/chat_app.py`, `_out/app_spec.json`

**실행 조건:** Session 3(키 불필요, LangGraph 설치 시)·Session 4 export(키 불필요)는 **오프라인 실행 가능** · Session 1·2 와 Session 4 스트리밍 스모크 확인은 **LLM 키 필요**. Session 2 는 최초 1회 한국어 임베딩 모델을 다운로드합니다.

In [ ]:
import json
import os
import time
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import logging
import warnings

# (en) Quiet library logs for readable notebook output.
# (kr) 노트북 출력을 읽기 쉽도록 라이브러리 로그를 줄입니다.
for _log_name in (
    "exaone",
    "exaone.llm",
    "exaone.llm.exaone_client",
    "urllib3",
    "transformers",
    "sentence_transformers",
):
    logging.getLogger(_log_name).setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="Unverified HTTPS request")
# (en) transformers/sentence-transformers use their own logging — quiet the first-load BertModel report.
# (kr) transformers/sentence-transformers 는 자체 로깅을 쓰므로 최초 로드 시 BertModel 리포트를 끕니다.
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")

# (en) Requires editable install at repo root: pip install -r requirements.txt && pip install -e ./exaone
# (kr) 저장소 루트에서 editable 설치가 필요합니다: pip install -r requirements.txt && pip install -e ./exaone
try:
    import exaone
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "exaone이 설치되지 않았습니다. 저장소 루트에서 "
        "pip install -r requirements.txt && pip install -e ./exaone 후 커널을 재시작하세요."
    ) from exc

exaone.load_project_env()
# (en) API_KEY gates live LLM steps AND is reused by the LangChain/LlamaIndex bridge clients below.
# (kr) API_KEY 는 실제 LLM 호출 여부를 결정하며, 아래 LangChain/LlamaIndex 브리지 클라이언트가 그대로 재사용합니다.
API_KEY = os.environ.get("EXAONE_API_KEY", "").strip()
HAS_API = bool(API_KEY)
ROOT = exaone.project_root()
TRACK09 = ROOT / "recipes" / "track09_framework_bridges"
DATA = TRACK09 / "data"
out_dir = Path("_out")
out_dir.mkdir(parents=True, exist_ok=True)

# (en) Shared fixtures from Track 04 (RAG snippets) and Track 06 (workflow brief).
# (kr) Track 04(RAG 스니펫)·Track 06(워크플로 브리프)의 공유 데이터를 재사용합니다.
MANUAL_PATH = (
    ROOT
    / "recipes"
    / "track04_rag_and_knowledge"
    / "data"
    / "internal_manual_snippets.json"
)
WORKFLOW_BRIEF = (
    ROOT
    / "recipes"
    / "track06_orchestration_multi_agent"
    / "data"
    / "workflow_brief.json"
)
POLICY_SNIPPETS = (
    ROOT
    / "recipes"
    / "track06_orchestration_multi_agent"
    / "data"
    / "policy_snippets.json"
)

BASE_URL = os.environ.get("EXAONE_BASE_URL", "").strip() or "http://localhost:8000/v1"
# (en) OpenAI-convention clients (LangChain/LlamaIndex) append /chat/completions themselves; strip it if the env
#      value already includes it, so the bridge base_url matches what those clients expect (native client does the same).
# (kr) OpenAI 규약 클라이언트(LangChain/LlamaIndex)는 /chat/completions 를 직접 붙입니다. env 값에 이미 포함돼 있으면
#      떼어내 브리지 base_url 을 맞춥니다(네이티브 클라이언트도 동일하게 정규화합니다).
BASE_URL = BASE_URL.rstrip("/")
if BASE_URL.endswith("/chat/completions"):
    BASE_URL = BASE_URL[: -len("/chat/completions")]
MODEL = (
    os.environ.get("EXAONE_MODEL", "").strip() or exaone.llm.ExaoneClient.DEFAULT_MODEL
)
exaone_client = exaone.integrations.build_llm_from_env() if HAS_API else None


# (en) Keyword-overlap ranker — counts query tokens that appear in each snippet body (no embedding server).
# (kr) 키워드 겹침 랭커 — 각 스니펫 본문에 등장하는 질의 토큰 수를 셉니다(임베딩 서버 불필요).
def pick_snippets(query: str, items: list[dict], top_k: int = 3) -> list[dict]:
    tokens = [t for t in query.replace("?", " ").split() if len(t) >= 2]
    scored = []
    for item in items:
        text = item["text"]
        # (en) Match against the snippet body only; adding `t in query` would tie every snippet.
        # (kr) 스니펫 본문만 대상으로 매칭합니다. `t in query` 를 더하면 모든 스니펫이 동점이 됩니다.
        score = sum(1 for t in tokens if t in text)
        scored.append((score, item))
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return [item for score, item in scored[:top_k] if score > 0] or items[:top_k]


def nfc(text: str) -> str:
    return unicodedata.normalize("NFC", text or "")


print("exaone", exaone.__version__, "| HAS_API =", HAS_API, "| model", MODEL)

**출력 해석:** `exaone <버전> | HAS_API = <값> | model <모델명>` 한 줄이 찍히면 기본 설정이 끝난 것입니다.
- `HAS_API`: 키가 있으면 `True` — Session 1·2 와 Session 4 스트리밍 스모크 확인이 실제로 실행됩니다(없으면 해당 단계만 건너뜁니다).
- `model`: 이후 모든 브리지가 호출할 EXAONE 모델 id 입니다(랭커 `pick_snippets`·공유 데이터 경로도 이 셀에서 준비됩니다).

## Session 1. LangChain — LCEL · 도구 호출

LangChain 의 ChatOpenAI 를 EXAONE 엔드포인트에 연결하고, LCEL 파이프(prompt parser)와 @tool/bind_tools 흐름을 확인합니다.


In [ ]:
HAS_LC = False
lc_llm = None
# (en) Catch only ImportError so config bugs surface instead of masquerading as "package unavailable".
# (kr) ImportError 만 잡아 설정 버그가 "패키지 없음"으로 위장되지 않게 합니다.
try:
    import httpx
    from langchain_openai import ChatOpenAI
except ImportError as exc:
    print("[SKIP] LangChain 미설치:", exc)
else:
    # (en) Mirror exaone.llm TLS policy for the LangChain httpx client.
    # (kr) LangChain httpx 클라이언트에 exaone.llm 과 동일한 TLS 정책을 적용합니다.
    _verify_ssl = os.environ.get("DISABLE_SSL_VERIFY", "").strip().lower() not in (
        "1",
        "true",
        "yes",
    )
    _extra_raw = os.environ.get("EXAONE_API_EXTRA_HEADERS", "").strip()
    _extra_headers = json.loads(_extra_raw) if _extra_raw else {}
    # (en) enable_thinking is non-standard for OpenAI clients — pass it via extra_body.chat_template_kwargs.
    # (kr) enable_thinking 은 OpenAI 클라이언트의 비표준 파라미터이므로 extra_body.chat_template_kwargs 로 넘깁니다.
    lc_llm = ChatOpenAI(
        default_headers=_extra_headers,
        base_url=BASE_URL,
        api_key=API_KEY or "placeholder",
        model=MODEL,
        temperature=1.0,
        top_p=0.95,
        max_tokens=1024,
        http_client=httpx.Client(verify=_verify_ssl),
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    HAS_LC = True
    print("LangChain model:", lc_llm.model_name)

**출력 해석:** `LangChain model: <모델명>` 이 보이면 `ChatOpenAI` 가 EXAONE 엔드포인트를 호출하도록 설정된 것입니다.
- `base_url` 을 EXAONE 으로 바꾸고 `extra_body={"chat_template_kwargs": {"enable_thinking": False}}` 로 thinking 채널을 끕니다(이 설정이 없으면 답변 앞에 추론 서두가 노출됩니다).
- LangChain 미설치 시에는 `[SKIP] LangChain 미설치` 가 찍히고 이후 단계는 건너뜁니다.

### Session 1-1. LCEL 번역 체인 + RAG (LangChain vs EXAONE-only)

**하는 일:** LCEL 파이프(`prompt | llm | parser`)로 한 줄 번역을 만들고, 검색된 스니펫을 근거로 LangChain·EXAONE-native 두 경로가 같은 질문에 답하도록 구성해 **출처 인용**을 비교합니다.

**핵심:** 두 경로 모두 같은 엔드포인트를 호출하므로 지연 시간(ms)은 잡음에 가깝습니다. 여기서 볼 것은 **인용한 출처**(근거 기반 답변인지)와 **추론 서두 누출 여부**입니다.

In [ ]:
manual = json.loads(MANUAL_PATH.read_text(encoding="utf-8"))
QUERY = manual["questions"][1]
snippets = manual["snippets"]
picked = pick_snippets(QUERY, snippets)
context_block = "\n\n".join(f"[{s['source']}] {s['text']}" for s in picked)
RAG_SYSTEM = "Answer in Korean using ONLY the provided context. Cite source tags like [it-vpn-guide]. Use polite Korean (높임말)."

lcel_sample = ""
rag_results = {
    "query": QUERY,
    "retrieved_sources": [s["source"] for s in picked],
    "langchain": {},
    "exaone_only": {},
}
print("질의:", QUERY)
print("검색된 출처:", rag_results["retrieved_sources"])

if HAS_LC and HAS_API:
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.messages import SystemMessage, HumanMessage

    # (en) LCEL pipe: prompt | llm | parser — StrOutputParser strips the chat wrapper to a plain string.
    # (kr) LCEL 파이프: prompt | llm | parser — StrOutputParser 가 채팅 래퍼를 평문 문자열로 변환합니다.
    translate_chain = (
        ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    "Translate into {target_lang}. Return only the translation.",
                ),
                ("human", "{text}"),
            ]
        )
        | lc_llm
        | StrOutputParser()
    )
    lcel_sample = nfc(
        translate_chain.invoke(
            {
                "text": "Please submit your leave request via the HR portal.",
                "target_lang": "Korean",
            }
        )
    )
    print("LCEL 번역:", lcel_sample[:120])

    t0 = time.perf_counter()
    lc_answer = nfc(
        lc_llm.invoke(
            [
                SystemMessage(content=f"{RAG_SYSTEM}\n\nContext:\n{context_block}"),
                HumanMessage(content=QUERY),
            ]
        ).content
    )
    lc_cites = sorted({s["source"] for s in picked if f"[{s['source']}]" in lc_answer})
    rag_results["langchain"] = {
        "latency_ms": round((time.perf_counter() - t0) * 1000, 1),
        "answer_chars": len(lc_answer),
        "cited_sources": lc_cites,
        "preview": lc_answer[:160],
    }

    opts = exaone.llm.ExaoneGenerateOptions(
        max_new_tokens=1024, temperature=1.0, top_p=0.95, enable_thinking=False
    )
    t1 = time.perf_counter()
    ex_answer = nfc(
        exaone_client.chat(
            messages=[
                exaone.llm.ExaoneMessage(
                    role="system", content=f"{RAG_SYSTEM}\n\nContext:\n{context_block}"
                ),
                exaone.llm.ExaoneMessage(role="user", content=QUERY),
            ],
            options=opts,
        ).content
    )
    ex_cites = sorted({s["source"] for s in picked if f"[{s['source']}]" in ex_answer})
    rag_results["exaone_only"] = {
        "latency_ms": round((time.perf_counter() - t1) * 1000, 1),
        "answer_chars": len(ex_answer),
        "cited_sources": ex_cites,
        "preview": ex_answer[:160],
    }
    print("LangChain 답변:", lc_answer[:140])
    print("인용한 출처 — LangChain:", lc_cites, "| EXAONE:", ex_cites)

**출력 해석:**
- **검색**: VPN 질문에 `검색된 출처: ['it-vpn-guide', 'it-vpn-guide']` — 랭커가 연차 정책이 아니라 VPN 가이드를 정확히 골랐습니다(키워드 랭커라 항상 같은 결과를 냅니다). 서로 다른 두 VPN 스니펫이 같은 출처 태그를 공유해 태그가 두 번 보이고, 인용 집합은 1개로 합쳐집니다.
- **LCEL 번역**: 추론 서두 없이 높임말 번역 한 줄만 나옵니다(예: "휴가 신청은 HR 포털을 통해 제출해 주세요." — 정확한 문구는 실행마다 다를 수 있습니다). 서두가 없다는 것은 `enable_thinking=False` 가 `extra_body` 로 잘 전달됐다는 신호입니다(Session 1-3 에서 그 대조를 직접 봅니다).
- **RAG 답변**: 두 경로 모두 `[it-vpn-guide]` 를 인용(`인용한 출처 — LangChain: ['it-vpn-guide'] | EXAONE: ['it-vpn-guide']`)하며 GlobalProtect·2FA 를 근거로 답합니다 — 브리지 경로도 네이티브 경로와 동등한 품질을 보입니다.
- 지연 시간(ms)은 같은 모델·엔드포인트를 쓰므로 의미 있는 비교 지표가 아닙니다(콘솔이 아니라 JSON 산출물에만 기록).

### Session 1-2. 도구 호출 (`@tool` + `bind_tools`)

**하는 일:** `@tool` 로 매뉴얼 검색 함수를 정의해 `bind_tools` 로 모델에 연결하고, 모델이 요청한 도구 호출을 **실제로 실행**한 뒤 결과를 `ToolMessage` 로 다시 전달해 최종 답변까지 한 턴을 완성합니다.

**핵심:** "모델이 어떤 도구를 어떤 인자로 호출하는가" → "그 결과로 무엇을 답하는가" 의 흐름을 봅니다.

In [ ]:
tool_trace = {}
if HAS_LC and HAS_API:
    from langchain_core.tools import tool
    from langchain_core.messages import HumanMessage, ToolMessage

    @tool
    def lookup_manual_topic(topic: str) -> str:
        """Look up internal manual snippets by topic keyword."""
        hits = pick_snippets(topic, snippets, top_k=2)
        return "\n".join(f"{h['source']}: {h['text']}" for h in hits)

    bound = lc_llm.bind_tools([lookup_manual_topic])
    question = "VPN 접속 조건을 매뉴얼에서 찾아주세요."
    ai_msg = bound.invoke([HumanMessage(content=question)])
    calls = ai_msg.tool_calls or []
    if calls:
        # (en) Execute each requested tool call, feed the result back as ToolMessage, get the grounded answer.
        # (kr) 요청된 도구 호출을 실행해 결과를 ToolMessage 로 다시 전달하고, 근거 있는 최종 답변을 받습니다.
        tool_msgs = [
            ToolMessage(
                content=lookup_manual_topic.invoke(c["args"]), tool_call_id=c["id"]
            )
            for c in calls
        ]
        final = nfc(
            bound.invoke([HumanMessage(content=question), ai_msg, *tool_msgs]).content
        )
        tool_trace = {
            "tool_calls": [{"name": c["name"], "args": c["args"]} for c in calls],
            "final_answer": final[:200],
        }
        print(
            "tool_calls:",
            [c["name"] for c in calls],
            "| args:",
            [c["args"] for c in calls],
        )
        print("최종 답변:", final[:120])
    else:
        tool_trace = {
            "tool_calls": [],
            "note": "모델이 도구를 호출하지 않고 바로 답했습니다(EXAONE 함수호출 포맷 확인 필요).",
        }
        print("[주의] tool_calls가 비었습니다 — 직접 답변:", (ai_msg.content or "")[:100])

payload = {
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "has_api": HAS_API,
    "has_langchain": HAS_LC,
    "model": MODEL,
    "lcel_sample": lcel_sample[:200],
    "rag_compare": rag_results,
    "tool_trace": tool_trace,
    "notes": "Bridge track — prefer exaone.agents ToolAgent for production harness.",
}
(out_dir / "langchain_vs_exaone.json").write_text(
    json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("saved:", (out_dir / "langchain_vs_exaone.json").resolve())

**출력 해석:**
- `tool_calls: ['lookup_manual_topic']` 와 인자(예: `{'topic': 'VPN 접속 조건'}`) — 모델이 자유 답변 대신 **도구를 호출**했고, 질문에서 추출한 `topic` 인자까지 넘겼습니다.
- `최종 답변:` 은 그 도구 결과(매뉴얼 스니펫)를 근거로 한 답입니다 — 호출 여부만 본 것이 아니라 **실행→결과 재전달→답변** 한 턴을 완주했습니다.
- 모델이 도구를 부르지 않으면 `[주의] tool_calls가 비었습니다` 로 표시되어 문제 없는 것처럼 지나가는 상황을 막습니다.
- `saved:` — `langchain_vs_exaone.json` 에 번역·RAG·도구 트레이스가 저장됩니다.

### Session 1-3. enable_thinking 누출 대조 (extra_body 가 필요한 이유)

**하는 일:** 같은 번역 요청을 `extra_body` **없이**(EXAONE thinking 기본 ON)와 **넣어서**(`enable_thinking=False`) 두 번 호출해, 추론 서두가 노출되는 경우와 사라지는 경우를 직접 비교합니다.

**핵심:** 표준 OpenAI 클라이언트는 EXAONE 비표준 파라미터를 조용히 누락할 수 있습니다 — 그 결과를 설명이 아니라 실제 출력으로 확인합니다.

In [ ]:
# (en) Contrast — the SAME translate call WITHOUT vs WITH the enable_thinking knob, to SHOW why extra_body matters.
# (kr) 대조 — 같은 번역 호출을 enable_thinking 설정이 없을 때/있을 때로 비교해 extra_body 가 왜 필요한지 직접 보여 줍니다.
if HAS_LC and HAS_API:
    from langchain_core.messages import HumanMessage

    _verify = os.environ.get("DISABLE_SSL_VERIFY", "").strip().lower() not in (
        "1",
        "true",
        "yes",
    )
    _prompt = [
        HumanMessage(
            content="다음 문장을 한국어로 번역만 해 주세요: 'Submit the report by Friday.'"
        )
    ]
    # (en) No extra_body → the OpenAI client sends no chat_template_kwargs → EXAONE thinking defaults ON.
    # (kr) extra_body 없음 → OpenAI 클라이언트가 chat_template_kwargs 를 보내지 않아 EXAONE thinking 기본 ON 이 적용됩니다.
    lc_thinking_on = ChatOpenAI(
        base_url=BASE_URL,
        api_key=API_KEY or "placeholder",
        model=MODEL,
        temperature=1.0,
        top_p=0.95,
        max_tokens=512,
        http_client=httpx.Client(verify=_verify),
    )
    leaked = nfc(lc_thinking_on.invoke(_prompt).content)
    clean = nfc(
        lc_llm.invoke(_prompt).content
    )  # lc_llm already passes enable_thinking=False via extra_body
    print("[enable_thinking 설정 없음 · 기본 ON]:", leaked[:110])
    print("[enable_thinking=False 적용   ]:", clean[:110])

**출력 해석:**
- **`enable_thinking` 설정 없음(기본 ON)**: 답변 앞에 영어 추론 서두가 노출됩니다(예: "Okay, the user wants me to translate ... Let me start by ..."). 표준 OpenAI 클라이언트가 EXAONE 의 `chat_template_kwargs` 를 보내지 않아 서버 기본값(thinking ON)이 적용된 결과입니다.
- **`extra_body` 적용(`enable_thinking=False`)**: 깔끔한 번역만 나옵니다(예: "금요일까지 보고서를 제출하세요.").
- 같은 모델·같은 프롬프트인데 결과가 갈리는 차이는 **비표준 파라미터 전달 여부** 하나뿐입니다 — 이것이 Session 1·2 가 `extra_body`/`additional_kwargs` 를 쓰는 이유입니다.
- (서버가 thinking 채널을 지원하지 않는 배포에서는 서두가 보이지 않을 수 있습니다.)

## Session 2. LlamaIndex — QueryEngine · ReActAgent

EXAONE 의 비표준 model id 를 LlamaIndex 가 받아들이도록 `OpenAICompatLLM`(쿡북 제공 어댑터)을 씁니다. 표준 OpenAI 클라이언트는 `top_p` 같은 필드만 받고 비표준 `enable_thinking` 은 누락할 수 있으므로, `additional_kwargs` 로 `top_p` 를, `extra_body` 로 `enable_thinking` 을 함께 전달합니다. 임베딩은 **한국어 문서**에 맞춰 다국어 모델(`multilingual-e5-small`)을 쓰며 최초 1회 다운로드됩니다(없으면 안전하게 건너뜁니다).

In [ ]:
HAS_LI = False
li_llm = None
# (en) Catch only ImportError so config bugs surface instead of looking like a missing package.
# (kr) ImportError 만 잡아 설정 버그가 패키지 누락처럼 보이지 않게 합니다.
try:
    import exaone.integrations.llamaindex_openai_compat as li_compat
except ImportError as exc:
    print("[SKIP] LlamaIndex 미설치:", exc)
else:
    _extra_raw = os.environ.get("EXAONE_API_EXTRA_HEADERS", "").strip()
    _extra_headers = json.loads(_extra_raw) if _extra_raw else {}
    # (en) OpenAI clients drop unknown fields: send top_p as a native param and enable_thinking via extra_body.
    # (kr) OpenAI 클라이언트는 알 수 없는 필드를 버리므로 top_p 는 네이티브 파라미터로, enable_thinking 은 extra_body 로 보냅니다.
    li_llm = li_compat.OpenAICompatLLM(
        default_headers=_extra_headers,
        model=MODEL,
        api_base=BASE_URL,
        api_key=API_KEY or "placeholder",
        temperature=1.0,
        max_tokens=1024,
        additional_kwargs={
            "top_p": 0.95,
            "extra_body": {"chat_template_kwargs": {"enable_thinking": False}},
        },
    )
    HAS_LI = True
    print("LlamaIndex LLM model:", li_llm.model)

**출력 해석:** `LlamaIndex LLM model: <모델명>` 이 보이면 `OpenAICompatLLM` 이 EXAONE 비표준 model id 를 받아들여 준비된 것입니다.
- 핵심은 `additional_kwargs={"top_p": 0.95, "extra_body": {"chat_template_kwargs": {"enable_thinking": False}}}` — LlamaIndex 는 `top_p` 를 직접 받지 않으므로 `additional_kwargs` 로, thinking 설정은 `extra_body` 로 넣어야 둘 다 실제 요청에 실립니다.
- LlamaIndex 미설치 시 `[SKIP] LlamaIndex 미설치` 후 다음 단계는 건너뜁니다.

### Session 2-1. In-memory 인덱스 + QueryEngine + (선택) ReActAgent

**하는 일:** 매뉴얼 스니펫으로 메모리 벡터 인덱스를 만들고 `QueryEngine` 으로 한국어 질문에 답한 뒤(검색 노드·점수 노출), `QueryEngineTool` 을 연결한 `ReActAgent` 로 한 턴을 실행합니다.

**핵심:** 검색이 **관련 노드를 점수와 함께** 가져오는지, ReAct 가 도구 호출을 거쳐 답하는지를 봅니다. (인덱스 빌드 실패 시 `[SKIP]`)

In [ ]:
INDEX_OK = False
query_engine = None
qe_row = {"ok": False}
react_row = {"attempted": False}

if HAS_LI:
    try:
        from llama_index.core import Document, VectorStoreIndex, Settings
        from llama_index.embeddings.huggingface import HuggingFaceEmbedding

        documents = [
            Document(
                text=s["text"], metadata={"source": s["source"], "doc_id": s["id"]}
            )
            for s in manual["snippets"]
        ]
        Settings.llm = li_llm
        # (en) Multilingual embedding — the manual snippets and questions are Korean (en-only would retrieve poorly).
        # (kr) 다국어 임베딩 — 매뉴얼 스니펫·질문이 한국어라 영어 전용 모델은 검색 품질이 떨어질 수 있습니다. (최초 1회 모델 다운로드)
        Settings.embed_model = HuggingFaceEmbedding(
            model_name="intfloat/multilingual-e5-small"
        )
        index = VectorStoreIndex.from_documents(documents, show_progress=False)
        query_engine = index.as_query_engine(similarity_top_k=3)
        INDEX_OK = True
        print("index docs:", len(documents))
    except Exception as exc:
        print("[SKIP] index build failed:", type(exc).__name__, str(exc)[:120])

if INDEX_OK and query_engine is not None and HAS_API:
    q0 = manual["questions"][0]
    t0 = time.perf_counter()
    resp = query_engine.query(q0)
    # (en) Surface which snippets the retriever actually returned, with similarity scores.
    # (kr) 리트리버가 실제로 가져온 스니펫과 유사도 점수를 보여 줍니다.
    sources = [
        {
            "source": n.metadata.get("source"),
            "score": round(n.score, 3) if n.score is not None else None,
        }
        for n in (resp.source_nodes or [])
    ]
    qe_row = {
        "ok": True,
        "query": q0,
        "latency_ms": round((time.perf_counter() - t0) * 1000, 1),
        "answer_preview": str(resp)[:200],
        "sources": sources,
    }
    print("QueryEngine 질문:", q0)
    print("QueryEngine 답변:", qe_row["answer_preview"][:120])
    print("검색 노드(source·score):", sources)

    try:
        from llama_index.core.agent.workflow import ReActAgent
        from llama_index.core.tools import QueryEngineTool

        qe_tool = QueryEngineTool.from_defaults(
            query_engine=query_engine,
            name="manual_search",
            description="Search internal HR/IT manual snippets.",
        )
        agent = ReActAgent(tools=[qe_tool], llm=li_llm, verbose=False)
        react_row = {"attempted": True}
        t0 = time.perf_counter()
        # (en) Top-level await — Jupyter/ipykernel already runs an event loop, so asyncio.run() would error.
        # (kr) top-level await — Jupyter/ipykernel 은 이미 이벤트 루프를 실행 중이므로 asyncio.run() 은 오류가 납니다.
        agent_out = await agent.run(user_msg="연차 신청은 어디에서 하나요?")
        # (en) agent_out.response is a ChatMessage; take .content so the role prefix ("assistant:") is not shown.
        # (kr) agent_out.response 는 ChatMessage 이므로 .content 만 가져와 역할 접두사("assistant:")가 노출되지 않게 합니다.
        _resp = getattr(agent_out, "response", agent_out)
        react_text = nfc(str(getattr(_resp, "content", _resp))).strip()
        react_row.update(
            {
                "ok": True,
                "latency_ms": round((time.perf_counter() - t0) * 1000, 1),
                "answer_preview": react_text[:200],
            }
        )
        print("ReAct 답변:", react_text[:120])
    except Exception as exc:
        react_row.update({"ok": False, "error": f"{type(exc).__name__}: {exc}"[:200]})
        print("[WARN] ReAct failed:", react_row.get("error", "")[:80])

payload = {
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "has_llamaindex": HAS_LI,
    "index_ok": INDEX_OK,
    "has_api": HAS_API,
    "query_engine": qe_row,
    "react": react_row,
}
(out_dir / "react_trace.json").write_text(
    json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("saved:", (out_dir / "react_trace.json").resolve())

**출력 해석:**
- `index docs: 8` — 8개 스니펫이 색인되었습니다. (키가 없으면 여기까지만 찍히고 아래 QueryEngine·ReAct 줄은 건너뜁니다.)
- **QueryEngine**: "연차는 입사 첫해에 며칠까지" 질문에 `최대 11일` 이라 답합니다(데이터에 근거한 고정 사실). `검색 노드(source·score)` 상위 2개는 `hr-leave-policy`(유사도 ~0.92·~0.84)로 관련성이 가장 높고, 3번째 `payroll-calendar`(~0.83)는 상대적으로 관련성이 낮은 노드입니다 — 점수 끝자리는 torch·하드웨어 환경마다 다를 수 있습니다. 다국어 임베딩을 사용했기 때문에 한국어 질의가 관련 문서를 잘 가져왔습니다(영어 전용 모델이면 정확도가 떨어질 수 있습니다).
- 답변에 추론 서두가 없습니다 — `extra_body` 로 넘긴 `enable_thinking=False` 가 LlamaIndex 경로에도 적용된 것입니다(Session 1-3 대조와 같은 원리).
- **ReAct**: 연차 신청 경로(HR 포털)를 도구 호출을 거쳐 답합니다(예: "연차 신청은 HR 포털에서 휴가 메뉴를 통해…" — 정확한 문구는 실행마다 다를 수 있습니다). `await agent.run(...)`(top-level await)로 한 턴을 완주했습니다(`asyncio.run` 은 Jupyter 이벤트 루프와 충돌).
- `saved:` — `react_trace.json` 에 QueryEngine 답변·검색 노드·ReAct 결과가 저장됩니다.

## Session 3. LangGraph — StateGraph (mock 노드, 키 불필요)

Track 06 의 planner→executor→critic 흐름을 LangGraph 의 **명시적 엣지**로 표현합니다. 노드는 결정적 mock 이라 LLM 없이 실행됩니다. 핵심 차이는 **루프의 위치**입니다: LangGraph 는 조건부 엣지로 critic→executor 재실행을 **그래프에 드러내고**, EXAONE-native ToolAgent 는 같은 루프를 **하니스 내부**에 둡니다. 마지막에 두 방식의 단계 방문 횟수를 비교합니다.

In [ ]:
HAS_LG = False
try:
    from typing import Literal, TypedDict
    from langgraph.graph import END, START, StateGraph

    HAS_LG = True
except ImportError as exc:
    print("[SKIP] LangGraph 미설치:", type(exc).__name__, str(exc)[:100])

final_state = {}
mapping = []

if HAS_LG:
    brief = json.loads(WORKFLOW_BRIEF.read_text(encoding="utf-8"))
    policy = json.loads(POLICY_SNIPPETS.read_text(encoding="utf-8"))

    class WorkflowState(TypedDict, total=False):
        task_title: str
        phase: str
        plan: dict
        draft: str
        review: dict
        stop: bool
        trace: list

    def planner_node(state):
        trace = list(state.get("trace") or [])
        trace.append({"node": "planner", "phase": "planner"})
        return {
            "phase": "planner",
            "plan": {
                "executor_brief": brief.get("user_goal", ""),
                "stop_when": "체크리스트 충족",
            },
            "trace": trace,
        }

    def executor_node(state):
        trace = list(state.get("trace") or [])
        rounds = sum(1 for t in trace if t["node"] == "executor")
        # (en) Round 0 covers only 재택 so the critic fails once; the next round widens coverage to close the loop.
        # (kr) 첫 실행에서는 재택만 다뤄 critic 이 한 번 실패하고, 다음 실행에서 범위를 넓혀 루프를 닫습니다.
        topics = ("재택",) if rounds == 0 else ("재택", "VPN", "연차", "보안")
        hits = []
        for topic in topics:
            hits.extend(
                [
                    s
                    for s in policy
                    if topic in s.get("topic", "") or topic in s.get("id", "")
                ]
            )
        draft = (
            "\n".join(
                f"- {h['text']} (source: {h.get('source', h.get('id'))})"
                for h in hits[:6]
            )
            or "(no hits)"
        )
        trace.append(
            {
                "node": "executor",
                "phase": "executor",
                "round": rounds,
                "hit_count": len(hits),
            }
        )
        return {"phase": "executor", "draft": draft, "trace": trace}

    def critic_node(state):
        checklist = brief.get("critic_checklist", [])
        draft = state.get("draft") or ""
        review = {
            item: any(token in draft for token in item.split()[:2])
            for item in checklist
        }
        passed = sum(1 for v in review.values() if v)
        trace = list(state.get("trace") or [])
        trace.append(
            {
                "node": "critic",
                "phase": "critic",
                "passed": passed,
                "total": len(checklist),
            }
        )
        # (en) Stop at all-but-one: the 보안 item has no matching keyword in the snippet body — a deliberate
        #      illustration of how brittle a keyword critic is (requiring a full pass would never converge here).
        # (kr) 전체 항목 중 하나를 제외한 수준에서 멈춥니다: 보안 항목은 스니펫 본문에 매칭 키워드가 없어
        #      키워드 크리틱의 취약성을 보여주는 의도된 예입니다(전체 통과를 요구하면 여기서는 수렴하지 않습니다).
        return {
            "phase": "critic",
            "review": review,
            "stop": passed >= max(1, len(checklist) - 1),
            "trace": trace,
        }

    def after_critic(state) -> "Literal['executor', 'done']":
        rounds = sum(1 for t in state.get("trace", []) if t["node"] == "executor")
        # (en) Loop back to executor until the checklist is nearly met (all-but-one), capped at 2 rounds.
        # (kr) 체크리스트가 거의 충족될 때까지 executor 로 되돌아가되, 종료 보장을 위해 2회차로 제한합니다.
        return "done" if (state.get("stop") or rounds >= 2) else "executor"

    builder = StateGraph(WorkflowState)
    builder.add_node("planner", planner_node)
    builder.add_node("executor", executor_node)
    builder.add_node("critic", critic_node)
    builder.add_edge(START, "planner")
    builder.add_edge("planner", "executor")
    builder.add_edge("executor", "critic")
    builder.add_conditional_edges(
        "critic", after_critic, {"executor": "executor", "done": END}
    )
    graph = builder.compile()
    final_state = graph.invoke({"task_title": brief["title"], "trace": []})
    print("nodes visited:", [t["node"] for t in final_state.get("trace", [])])

    # (en) Compare against the EXAONE-native phase order (Track 06 06_orchestration_lab): planner→executor→critic.
    # (kr) EXAONE-native 단계 순서(Track 06)와 비교합니다: planner→executor→critic.
    # (en) Track 06 runs each phase once and keeps the retry loop INSIDE the ToolAgent; LangGraph makes the loop
    #      explicit, so executor/critic appear more than once here. The visit counts make that difference visible.
    # (kr) Track 06 은 각 단계를 한 번씩 실행하고 재시도 루프를 ToolAgent 내부에 둡니다. LangGraph 는 루프를 명시적
    #      엣지로 드러내므로 executor/critic 이 여러 번 나타납니다. 방문 횟수로 그 차이를 확인합니다.
    EXAONE_NATIVE_PHASES = ["planner", "executor", "critic"]
    graph_phases = [row["phase"] for row in final_state.get("trace", [])]
    mapping = [
        {
            "phase": p,
            "native_visits": 1,
            "langgraph_visits": graph_phases.count(p),
            "differs": graph_phases.count(p) != 1,
        }
        for p in EXAONE_NATIVE_PHASES
    ]
    print("phase mapping (native 1회 vs LangGraph 방문수):", mapping)

    payload = {
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "task_id": brief.get("task_id"),
        "langgraph_trace": final_state.get("trace", []),
        "phase_mapping": mapping,
        "critic_review": final_state.get("review", {}),
        "notes": "LangGraph makes the planner→executor→critic loop explicit via conditional edges; exaone.agents keeps that loop inside the ToolAgent harness.",
    }
    (out_dir / "langgraph_vs_exaone.json").write_text(
        json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print("saved:", (out_dir / "langgraph_vs_exaone.json").resolve())

**출력 해석:**
- `nodes visited: ['planner', 'executor', 'critic', 'executor', 'critic']` — critic 이 첫 초안(재택만 다룸)을 통과시키지 못해 **executor 로 되돌아가 다시 실행**했습니다. 조건부 엣지(루프)가 실제로 실행된 것입니다.
- 2회차에서도 `보안` 항목은 false 로 남습니다(`보안 사고` 키워드가 sec-incident 본문에 없어 단순 키워드 크리틱이 놓침) — critic 은 4개 중 3개 항목 충족으로 종료합니다. **키워드 기반 크리틱의 한계**를 보여주는 의도된 결과입니다.
- `phase mapping`: `planner` 는 1회로 같지만 `executor`·`critic` 은 LangGraph 에서 2회(`differs: True`) — LangGraph 는 루프를 그래프에 명시해 단계가 여러 번 보이고, EXAONE-native 는 그 루프를 ToolAgent 내부에 숨겨 1회만 보입니다.
- `saved:` — `langgraph_vs_exaone.json` 에 노드 trace·방문 횟수 비교·critic 리뷰(보안 false 포함)가 저장됩니다(키 없이 생성).

## Session 4. Gradio — 채팅 UI 앱

EXAONE 을 Gradio 채팅 UI 로 감싸는 핵심은 **`chat_stream` 을 `gr.ChatInterface` 가 받을 수 있는 제너레이터로 바꾸는 것** 하나입니다. `data/chat_app_template.py` 의 `stream_chat` 이 그 연결 방식입니다:

```python
for event in client.chat_stream(messages=messages, options=stream_opts):
    delta = event.text if getattr(event, "kind", None) == "text" else ""
    if delta:
        chunks.append(delta)
        yield normalize_unicode("".join(chunks))  # 누적 텍스트를 매 토큰마다 반환하여 UI를 실시간 갱신
```

이 노트북은 두 가지를 합니다 — (1) 그 템플릿을 **배포용 `_out/chat_app.py` 로 export** 하고 스트리밍 스모크 확인으로 연결을 점검하고, (2) 다음 셀에서 **노트북 안에서 바로 앱을 실행해**(`LAUNCH_UI`) 브라우저로 열어 볼 수 있게 합니다. export·스모크 확인은 자동 실행되고, 라이브 앱은 `LAUNCH_UI=True` 일 때만 실행됩니다(노트북에서 `launch()` 는 블로킹이라 기본은 꺼둡니다).

In [ ]:
import ast

# (en) Optional streaming smoke — confirms chat_stream wiring before launching the UI (needs a key).
# (kr) 선택적 스트리밍 스모크 확인 — UI 실행 전 chat_stream 연결을 확인합니다(키 필요).
stream_preview = ""
if HAS_API and exaone_client is not None:
    stream_opts = exaone.llm.ExaoneGenerateOptions(
        max_new_tokens=64, temperature=1.0, enable_thinking=False
    )
    chunks = []
    # (en) Print each delta as it arrives so token-by-token streaming is actually visible (not just the final text).
    # (kr) 도착하는 델타를 즉시 출력해 최종 텍스트뿐 아니라 토큰 단위 스트리밍도 보이게 합니다.
    print("stream preview:", end=" ", flush=True)
    for event in exaone_client.chat_stream(
        messages=[
            exaone.llm.ExaoneMessage(
                role="user", content="Say hello in one short Korean sentence."
            )
        ],
        options=stream_opts,
    ):
        if getattr(event, "kind", None) == "text":
            delta = getattr(event, "text", "") or ""
        elif isinstance(event, dict):
            delta = event.get("text") or event.get("delta") or ""
        else:
            delta = ""
        if delta:
            chunks.append(delta)
            print(delta, end="", flush=True)
    print()
    stream_preview = nfc("".join(chunks))
else:
    print("[SKIP] 스트리밍 스모크 확인 — 키가 없어 건너뜁니다(export 는 계속 진행).")

# (en) Export the app source and validate it parses — runs unconditionally (no key/Gradio needed here).
# (kr) 앱 소스를 export 하고 문법이 유효한지 검증합니다. 키·Gradio 없이 항상 실행됩니다.
APP_SOURCE = (DATA / "chat_app_template.py").read_text(encoding="utf-8")
ast.parse(APP_SOURCE)
app_path = out_dir / "chat_app.py"
app_path.write_text(APP_SOURCE, encoding="utf-8")

spec = {
    "framework": "gradio",
    "entrypoint": "python chat_app.py",
    "streaming": "exaone.llm.ExaoneAPIClient.chat_stream",
    "ui": "gr.ChatInterface (messages history format, gradio 6)",
    "parameters": ["system_prompt", "temperature", "max_tokens"],
    "launch_in_notebook": False,
    "smoke_tested": bool(stream_preview),
    "stream_preview_chars": len(stream_preview),
}
(out_dir / "app_spec.json").write_text(
    json.dumps(spec, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("exported:", app_path.resolve())
print("Run locally: python", app_path.resolve())

**출력 해석:**
- `stream preview:` 에 한 문장 한국어 인사가 토큰 단위로 출력됩니다(예: "안녕하세요!") — `chat_stream` 이 UI 에 보낼 텍스트 델타를 확인한 것입니다(키가 없으면 이 줄 대신 `[SKIP]`).
- `exported: …/chat_app.py` + `Run locally: python …` — 앱 소스가 문법 검증을 통과해 실행 파일로 저장됐습니다. 터미널에서 `python _out/chat_app.py` 로 실행하면 Gradio UI 가 열립니다(노트북에서 `launch()` 하면 셀이 멈추므로 여기서는 export 만 합니다).
- `app_spec.json` 의 `smoke_tested` 로 스트리밍 확인 여부를, `stream_preview_chars` 로 미리보기 길이를 기록합니다.

### Session 4-1. 노트북에서 앱 실행하기 (선택)

`LAUNCH_UI = True` 로 바꿔 아래 셀을 실행하면 export 한 앱을 **노트북에서 바로 실행하고** 로컬 URL 을 출력합니다 — 그 URL 을 클릭하면 브라우저에서 채팅할 수 있습니다. `prevent_thread_lock=True` 라 셀이 멈추지 않고(논블로킹) 바로 넘어갑니다. 종료하려면 `demo.close()` 를 호출합니다.

> 자동 검수(`nbconvert`)가 서버를 실행하지 않도록 기본값은 `LAUNCH_UI=False` 입니다.

In [ ]:
# (en) Interactive launch — run it yourself with LAUNCH_UI=True (kept off so nbconvert review stays server-free).
# (kr) 인터랙티브 실행 — LAUNCH_UI=True 로 직접 실행합니다(검수 자동 실행은 서버 없이 통과하도록 기본값을 꺼 둡니다).
LAUNCH_UI = False  # (en) set True to start the live app / (kr) True 로 바꾸면 앱이 실행되고 URL 이 출력됩니다

if LAUNCH_UI:
    import importlib.util

    # (en) Reuse the exported app source as the single source of truth (no duplicated UI code here).
    # (kr) export 한 앱 소스를 단일 소스로 그대로 재사용합니다(여기서 UI 코드를 중복하지 않습니다).
    _spec = importlib.util.spec_from_file_location(
        "track09_chat_app", DATA / "chat_app_template.py"
    )
    _app_mod = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(_app_mod)
    demo = _app_mod.build_demo()
    # (en) prevent_thread_lock=True returns immediately (non-blocking); print the local URL to click.
    # (kr) prevent_thread_lock=True 면 즉시 반환됩니다(논블로킹). 클릭할 로컬 URL 을 출력합니다.
    launched = demo.launch(prevent_thread_lock=True, inline=False, quiet=True)
    local_url = getattr(demo, "local_url", None) or (
        launched[1]
        if isinstance(launched, tuple) and len(launched) > 1
        else "http://127.0.0.1:7860"
    )
    print("브라우저에서 열기 →", local_url)
    print("종료하려면 → 바로 아래 '서버 종료' 셀을 실행하세요 (또는 새 셀에서 demo.close()).")
else:
    print(
        "[건너뜀] LAUNCH_UI=True 로 바꾸면 Gradio 앱을 실행하고 URL 을 출력합니다(브라우저에서 채팅 가능)."
    )

**출력 해석:**
- 기본(`LAUNCH_UI=False`): `[건너뜀] …` 만 출력되고 서버는 실행되지 않습니다 — 자동 검수가 문제 없이 통과합니다.
- `LAUNCH_UI=True`: `브라우저에서 열기 → http://127.0.0.1:7860` 같은 URL 이 출력됩니다. 클릭하면 Gradio 채팅 UI 가 브라우저에서 열리고 EXAONE 이 토큰 단위로 스트리밍 응답합니다. `prevent_thread_lock=True` 라 셀은 멈추지 않으며 `demo.close()` 로 종료합니다.
- **종료**: 바로 아래 **'서버 종료' 셀**을 실행하면 됩니다 — `demo` 가 있으면 `demo.close()` 로 서버를 내리고, 없으면(LAUNCH_UI=False) 안전하게 넘어갑니다.

In [ ]:
# (en) Stop the background Gradio server started above — safe to run anytime (no-op if nothing is running).
# (kr) 위에서 실행한 백그라운드 Gradio 서버를 종료합니다. 아무 때나 실행해도 안전합니다(없으면 아무 일도 하지 않습니다).
if "demo" in globals():
    demo.close()
    print("Gradio 서버를 종료했습니다.")
else:
    print("종료할 서버가 없습니다 (LAUNCH_UI=False 였거나 아직 실행하지 않음).")

## 체크포인트

- [ ] Session 1 `langchain_vs_exaone.json` — LCEL 번역·RAG 인용 출처·도구 호출 트레이스 (**키 필요**).
- [ ] Session 2 `react_trace.json` — QueryEngine 검색 노드(점수 포함)+ ReAct 1턴 (**키 필요**, 임베딩 최초 1회 다운로드).
- [ ] Session 3 `langgraph_vs_exaone.json` — planner/executor/critic 노드 trace + EXAONE 단계 방문수 비교 (**키 없이**, LangGraph 설치 시).
- [ ] Session 4 `chat_app.py` + `app_spec.json` — `ast.parse` 통과, 실행 가능한 앱 export (**키 없이**); `LAUNCH_UI=True` 로 노트북에서 바로 실행하고 URL 로 접속(스트리밍 스모크 확인·라이브 앱은 키 필요).

> 학습 포인트: 지연(ms)이 아니라 **브리지가 EXAONE 비표준 파라미터(`enable_thinking`·`top_p`)를 제대로 전달하는가**(추론 서두 누출·출처 인용)를 보세요.

## 마무리

이 노트북에서 EXAONE 을 네 프레임워크에 **최소 어댑터로** 연결해 실제로 실행해 보았습니다.

**시연한 것**
- **LangChain**: LCEL 번역(높임말, 추론 서두 없음)·RAG 답변이 `[it-vpn-guide]` 인용·도구 호출 1턴 완주.
- **LlamaIndex**: 다국어 임베딩으로 한국어 검색(노드+점수)·ReAct 1턴.
- **LangGraph**: critic→executor 루프가 실제로 실행되며(`executor`/`critic` 2회), EXAONE-native 의 내부 루프(1회)와 대비.
- **Gradio**: `chat_stream` 기반 채팅 앱을 노트북에서 바로 실행해(URL 클릭) 확인하고, 배포용 실행 파일로도 export.

**핵심 교훈과 한계**
- EXAONE 은 OpenAI 호환이라 `base_url` 교체로 대부분 연결되지만, **비표준 파라미터(`enable_thinking`·`top_p`)는 `extra_body`/`additional_kwargs` 로 명시 전달**해야 합니다 — 그렇지 않으면 추론 서두가 노출되거나 샘플링 규약이 조용히 무시됩니다.
- `base_url` 은 OpenAI 규약(`/v1` 으로 끝나는 형태)을 따라야 합니다 — 표준 클라이언트는 `/chat/completions` 를 자동으로 덧붙입니다.
- 이 브리지들은 **연결 패턴 학습용**입니다. 출시 하니스는 Track 10 처럼 **EXAONE-native ToolAgent** 를 권장합니다(관측·재시도·구조화 출력이 한 곳에 모입니다).

**다음 트랙**
- **Track 10 — AX Capstones**: EXAONE-native 하니스로 단일 에이전트 시스템을 처음부터 끝까지 구현합니다.
- (먼저 평가 체계가 필요하면 **Track 08 — Evaluation** 을 권장합니다.)